# Week 6 — Learning Which Contracts Are Similar

## Overview

**Research question:** Can simulated liability responses teach a distance to retrieve more useful reference contracts?

Euclidean distance treats equally scaled coordinates equally. We now fit one nonnegative weight per feature so that squared feature distances approximately match squared differences in reference liabilities. We then copy the value of the nearest reference under that learned distance.

This is a deliberately small, interpretable research construction. It is a way to investigate response-aware similarity without implementing a large metric-learning system or a fair-fee solver.

**Learning objectives**

- Build response-labelled pairs from reference contracts only.
- Explain a diagonal weighted distance and when it is a pseudometric.
- Derive the effect of MC noise on squared response gaps.
- Tune regularization using held-out validation MC labels.
- Compare learned neighbor transfer with Euclidean neighbors and direct predictors.

**Prerequisites:** Weeks 1–5; squared error, regularization, validation, and basic matrix notation. The constrained optimization helper is included; deriving an optimizer is optional.

**Output contract:** a regularization table, learned-weight plot, validation comparison, nearest-reference case study, and complete simulation/fitting ledger.

**Suggested workload:** 75-minute meeting plus 4–6 independent hours. The negative-control and label-noise exercises are core; the hybrid kNN exercise is optional.

**Study boundary:** fit pairs on references only; select settings on validation only. The test set stays closed through Week 7 and is opened only for the frozen Week 8 assessment.

## 1. A distance that responds to the pricing task

Let \(z(x)\in\mathbb R^p\) contain the candidate-scaled features, with \(p=5\). Define

\[
d_w(x,x')
=\left(\sum_{k=1}^{p}w_k[z_k(x)-z_k(x')]^2\right)^{1/2},
\qquad w_k\geq 0,\quad \sum_{k=1}^{p}w_k=p.
\]

This is ordinary Euclidean distance after multiplying coordinate \(k\) by \(\sqrt{w_k}\). With \(w=(1,\ldots,1)\), we recover the Euclidean baseline. Fixing the sum removes a meaningless overall scale: multiplying every weight by the same positive constant would not change the nearest-neighbor ranking.

The diagonal matrix \(W=\mathrm{diag}(w)\) is positive semidefinite. If all weights are strictly positive, \(d_w\) is a metric on the feature vectors. If a weight is zero, distinct vectors can have zero distance, so we call it a **pseudometric**. Even with positive weights, contracts differing only in excluded fields can share a feature vector; on full contract records the feature-based distance may therefore also be a pseudometric. The squared quantity \(d_w^2\) itself is not generally a metric because it need not satisfy the triangle inequality.

**Worked example.** A query is \(q=(0.5,0.5)\), with references \(A=(0.6,0.2)\) and \(B=(0.8,0.5)\). Euclidean distance prefers \(B\). With weights \((1.8,0.2)\), the first coordinate matters more and the selected neighbor changes to \(A\). This shows how weighting changes retrieval; without the query's response it does not show which neighbor is better for pricing.

In [ ]:
# Complete common pricing code. No data files or earlier notebooks are required.
import sys
import json
import platform
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

try:
    from IPython.display import display
except ImportError:
    def display(*objects):
        for obj in objects:
            print(obj)

RISK_FREE_RATE = 0.03
FEATURES = ['account_ratio', 'guarantee_ratio', 'maturity', 'volatility', 'fee']
plt.rcParams.update({'figure.figsize': (7.2, 4.3), 'axes.grid': True,
                     'grid.alpha': 0.22, 'font.size': 10})


def make_portfolio(n=1200, seed=2027):
    """Synthetic in-force GMAB contracts, never customer data.

    P is a reference premium, A0=P*account_ratio, and G=P*guarantee_ratio.
    T is the remaining term. All contracts share the same risk-free rate.
    Volatility describes the contract's fund; rates are annual decimals.
    """
    if not isinstance(n, (int, np.integer)) or n < 10:
        raise ValueError('n must be an integer of at least 10.')
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        'contract_id': np.arange(n, dtype=int),
        'premium': rng.uniform(10_000.0, 50_000.0, n),
        'account_ratio': rng.uniform(0.65, 1.40, n),
        'guarantee_ratio': rng.uniform(0.90, 1.60, n),
        'maturity': rng.integers(5, 21, n).astype(float),
        'volatility': rng.uniform(0.12, 0.35, n),
        'fee': rng.uniform(0.005, 0.030, n),
    })


def validate_contracts(df):
    required = ['contract_id', 'premium'] + FEATURES
    missing = [c for c in required if c not in df]
    if missing:
        raise ValueError(f'Missing contract columns: {missing}')
    if len(df) == 0:
        raise ValueError('Supply at least one contract.')
    values = df[required].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError('All contract fields must be finite.')
    ids = df['contract_id'].to_numpy(dtype=float)
    if (ids < 0).any() or (ids != np.floor(ids)).any() or len(np.unique(ids)) != len(ids):
        raise ValueError('contract_id must contain unique nonnegative integers.')
    if (df[['premium', 'account_ratio', 'guarantee_ratio']] <= 0).any().any():
        raise ValueError('Premium and account/guarantee ratios must be positive.')
    if (df[['maturity', 'volatility', 'fee']] < 0).any().any():
        raise ValueError('Maturity, volatility and fee must be nonnegative.')


def gmab_mc(df, n_paths=2048, seed=123):
    """Independent plain Monte Carlo for per-premium gross top-up liability.

    Identical (seed, contract_id) reproduces the same path stream even when
    the contract appears in a different subset or row order. Distinct IDs
    give independent streams. This is NOT a common-market portfolio-risk
    simulator: it estimates a sum of expected liabilities, not portfolio VaR.
    SE describes Monte Carlo sampling uncertainty only.
    """
    validate_contracts(df)
    if not isinstance(n_paths, (int, np.integer)) or n_paths < 2:
        raise ValueError('n_paths must be an integer of at least 2.')
    if not isinstance(seed, (int, np.integer)) or seed < 0:
        raise ValueError('seed must be a nonnegative integer.')
    values, errors, sds = [], [], []
    for row in df.itertuples():
        rng = np.random.default_rng(np.random.SeedSequence([int(seed), int(row.contract_id)]))
        z = rng.standard_normal(n_paths)
        terminal_ratio = row.account_ratio * np.exp(
            (RISK_FREE_RATE - row.fee - 0.5 * row.volatility**2) * row.maturity
            + row.volatility * np.sqrt(row.maturity) * z
        )
        payoff = np.exp(-RISK_FREE_RATE * row.maturity) * np.maximum(
            row.guarantee_ratio - terminal_ratio, 0.0)
        sd = float(np.std(payoff, ddof=1))
        values.append(float(np.mean(payoff)))
        errors.append(sd / np.sqrt(n_paths))
        sds.append(sd)
    return pd.DataFrame({'value': values, 'se': errors, 'payoff_sd': sds,
                         'paths': int(n_paths)}, index=df.index)


def gmab_exact(df):
    """Closed-form put benchmark under the simplified model: audit only.

    Never use these values as training or tuning labels in the project.
    The deterministic cases T=0 or sigma=0 are handled separately.
    """
    validate_contracts(df)
    a = df['account_ratio'].to_numpy(dtype=float)
    g = df['guarantee_ratio'].to_numpy(dtype=float)
    t = df['maturity'].to_numpy(dtype=float)
    sig = df['volatility'].to_numpy(dtype=float)
    fee = df['fee'].to_numpy(dtype=float)
    ans = np.maximum(g * np.exp(-RISK_FREE_RATE*t) - a * np.exp(-fee*t), 0.0)
    regular = (t > 0) & (sig > 0)
    if np.any(regular):
        tr, sr = t[regular], sig[regular]
        d1 = (np.log(a[regular] / g[regular]) +
              (RISK_FREE_RATE-fee[regular]+0.5*sr**2)*tr) / (sr*np.sqrt(tr))
        d2 = d1 - sr*np.sqrt(tr)
        ans[regular] = (g[regular]*np.exp(-RISK_FREE_RATE*tr)*norm.cdf(-d2)
                        - a[regular]*np.exp(-fee[regular]*tr)*norm.cdf(-d1))
    return np.maximum(ans, 0.0)


def split_portfolio(df, seed=2027):
    """Split contracts before building pairs, selecting labels or tuning."""
    validate_contracts(df)
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(df))
    n_candidate, n_validation = int(0.60*len(df)), int(0.20*len(df))
    if min(n_candidate, n_validation, len(df)-n_candidate-n_validation) < 1:
        raise ValueError('Portfolio is too small for the requested split.')
    return {
        'candidate': df.iloc[order[:n_candidate]].copy(),
        'validation': df.iloc[order[n_candidate:n_candidate+n_validation]].copy(),
        'test': df.iloc[order[n_candidate+n_validation:]].copy(),
    }


def score_predictions(y_true, y_pred, premium=None):
    """Errors per premium, plus signed aggregate relative error when requested."""
    truth = np.asarray(y_true, dtype=float).reshape(-1)
    pred = np.asarray(y_pred, dtype=float).reshape(-1)
    if truth.shape != pred.shape or len(truth) == 0:
        raise ValueError('Truth and predictions must have equal nonzero length.')
    if not (np.isfinite(truth).all() and np.isfinite(pred).all()):
        raise ValueError('Truth and predictions must be finite.')
    error = pred-truth
    out = {'rmse': float(np.sqrt(np.mean(error**2))),
           'mae': float(np.mean(np.abs(error))),
           'p95_abs': float(np.quantile(np.abs(error), 0.95)),
           'bias': float(np.mean(error)),
           'negative_fraction': float(np.mean(pred < 0))}
    if premium is not None:
        p = np.asarray(premium, dtype=float).reshape(-1)
        if p.shape != truth.shape or not np.isfinite(p).all() or np.any(p <= 0):
            raise ValueError('Premium weights must be finite, positive and aligned.')
        denominator = float(p @ truth)
        out['portfolio_relative_error'] = (float(p @ error)/denominator
                                            if abs(denominator) > 1e-12 else np.nan)
    return out


In [ ]:
# Complete shared learning code. All labels are supplied explicitly by the caller.
from scipy.spatial.distance import cdist
from scipy.optimize import minimize
from sklearn.preprocessing import MinMaxScaler, SplineTransformer
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import make_pipeline


def build_features(candidate, query=None, columns=FEATURES):
    """Fit feature scales only on the unlabelled candidate pool.

    A constant training column maps to zero on training data. A held-out
    observation outside the training range may map outside [0,1]; no clipping.
    No premium, contract ID or computed valuation is an informative feature.
    """
    scaler = MinMaxScaler()
    z_candidate = scaler.fit_transform(candidate[list(columns)])
    z_query = None if query is None else scaler.transform(query[list(columns)])
    return scaler, z_candidate, z_query


def select_representatives(Z, m, method='kmeans', seed=2027):
    """Return m DISTINCT row positions of actual candidate contracts.

    K-means is fitted to candidate FEATURES only. For each nonempty cluster,
    choose the member closest to that cluster's center. This is a member
    representative, not an exact k-medoids optimization or Feng et al.'s SRSC.
    """
    z = np.asarray(Z, dtype=float)
    if z.ndim != 2 or not np.isfinite(z).all():
        raise ValueError('Z must be a finite 2D matrix.')
    if not isinstance(m, (int, np.integer)) or not 1 <= m <= len(z):
        raise ValueError('m must lie between 1 and the number of candidates.')
    rng = np.random.default_rng(seed)
    if method == 'random':
        return np.sort(rng.choice(len(z), size=m, replace=False))
    if method != 'kmeans':
        raise ValueError("method must be 'random' or 'kmeans'.")
    km = KMeans(n_clusters=m, n_init=5, max_iter=150, random_state=seed)
    membership = km.fit_predict(z)
    chosen = []
    for cluster in range(m):
        members = np.flatnonzero(membership == cluster)
        if len(members):
            distances = np.sum((z[members] - km.cluster_centers_[cluster])**2, axis=1)
            chosen.append(int(members[np.argmin(distances)]))
    # Duplicate feature vectors can yield fewer than m nonempty clusters.
    # Keep the requested contract count without ever repeating an ID.
    if len(chosen) < m:
        remaining = np.setdiff1d(np.arange(len(z)), chosen)
        chosen.extend(rng.choice(remaining, m-len(chosen), replace=False).tolist())
    return np.sort(np.asarray(chosen, dtype=int))


def nearest_predict(Zref, yref, Zquery, weights=None, k=1):
    """Uniform k-neighbor transfer; k=1 is the CSM-style teaching baseline.

    A diagonal weighted distance is ordinary Euclidean distance after each
    coordinate is multiplied by sqrt(weight). Stable sort breaks exact ties
    in reference-row order. Chunking avoids an N-by-N distance matrix.
    """
    zr, zq = np.asarray(Zref, dtype=float), np.asarray(Zquery, dtype=float)
    yr = np.asarray(yref, dtype=float).reshape(-1)
    if zr.ndim != 2 or zq.ndim != 2 or zr.shape[1] != zq.shape[1] or len(zr) != len(yr):
        raise ValueError('Reference/query dimensions or labels do not align.')
    if not (np.isfinite(zr).all() and np.isfinite(zq).all() and np.isfinite(yr).all()):
        raise ValueError('Features and labels must be finite.')
    if not isinstance(k, (int, np.integer)) or not 1 <= k <= len(zr):
        raise ValueError('k must be between 1 and the reference count.')
    w = np.ones(zr.shape[1]) if weights is None else np.asarray(weights, dtype=float)
    if w.shape != (zr.shape[1],) or not np.isfinite(w).all() or np.any(w < 0):
        raise ValueError('Distance weights must be finite and nonnegative.')
    scale = np.sqrt(w)
    neighbors = np.empty((len(zq), k), dtype=int)
    for start in range(0, len(zq), 1024):
        stop = min(start+1024, len(zq))
        distance = cdist(zq[start:stop]*scale, zr*scale, metric='sqeuclidean')
        neighbors[start:stop] = np.argsort(distance, axis=1, kind='stable')[:, :k]
    return yr[neighbors].mean(axis=1), neighbors


def fit_metric(Zref, yref, lam=0.01, max_pairs=2000, seed=2027):
    """Fit a simple diagonal response-aware pseudometric on reference pairs.

    Objective: mean(((H@w)/p - ((y_i-y_j)/s)^2)^2)
               + lam/2 * sum((w-1)^2),  w>=0, sum(w)=p.
    Here H=(z_i-z_j)^2 coordinatewise and s=max(ptp(yref),1e-6).
    The label scale is part of the modeling choice, fitted on references only.
    Numerical labels have noise: this loss is not an unbiased exact-gap loss.
    """
    z, y = np.asarray(Zref, dtype=float), np.asarray(yref, dtype=float).reshape(-1)
    if z.ndim != 2 or len(z) != len(y) or len(y) < 2:
        raise ValueError('Supply at least two aligned labeled reference contracts.')
    if not (np.isfinite(z).all() and np.isfinite(y).all()):
        raise ValueError('Training data must be finite.')
    if lam < 0 or not isinstance(max_pairs, (int, np.integer)) or max_pairs < 1:
        raise ValueError('lam must be nonnegative and max_pairs a positive integer.')
    n, p = z.shape
    count = min(max_pairs, n*(n-1)//2)
    rng = np.random.default_rng(seed)
    if count == n*(n-1)//2:
        i, j = np.triu_indices(n, k=1)
    else:
        # Uniform directed non-self pairs sampled WITH replacement.
        # They share contracts and are not independent observational units.
        i = rng.integers(0, n, size=count)
        j = (i+rng.integers(1, n, size=count)) % n
    label_scale = max(float(np.ptp(y)), 1e-6)
    h = (z[i]-z[j])**2 / p
    target = ((y[i]-y[j])/label_scale)**2
    def objective(w):
        residual = h @ w-target
        return float(np.mean(residual**2) + 0.5*lam*np.sum((w-1.0)**2))
    def gradient(w):
        return 2.0*h.T @ (h @ w-target)/len(target) + lam*(w-1.0)
    result = minimize(objective, np.ones(p), jac=gradient, method='SLSQP',
                      bounds=[(0.0, float(p))]*p,
                      constraints=[{'type': 'eq', 'fun': lambda w: w.sum()-p,
                                    'jac': lambda w: np.ones(p)}],
                      options={'maxiter': 400, 'ftol': 1e-10})
    feasible = (np.isfinite(result.x).all() and result.x.min() >= -1e-7
                and abs(result.x.sum()-p) <= 1e-6)
    success = bool(result.success and feasible)
    weights = np.clip(result.x, 0.0, None) if success else np.ones(p)
    weights *= p/weights.sum()
    return {'weights': weights, 'success': success, 'pair_count': len(target),
            'objective': objective(weights), 'scale': label_scale,
            'iterations': int(result.nit), 'message': str(result.message)}


def make_regressors(seed=2027):
    """Fixed small model menu. Regression predicts values, not distances."""
    return {
        'Ridge': Ridge(alpha=1.0),
        'SplineGAM': make_pipeline(
            SplineTransformer(n_knots=4, degree=3, include_bias=False),
            Ridge(alpha=1.0)),
        'RandomForest': RandomForestRegressor(n_estimators=100, min_samples_leaf=3,
                                               random_state=seed, n_jobs=1),
        'GradientBoosting': GradientBoostingRegressor(n_estimators=100, max_depth=2,
                                                       learning_rate=0.05, random_state=seed),
    }


In [ ]:
from sklearn.base import clone

toy_query = np.array([[0.5, 0.5]])
toy_references = np.array([[0.6, 0.2], [0.8, 0.5]])
toy_weights = np.array([1.8, 0.2])
toy_labels = np.array([0.05, 0.12])  # Arithmetic example, not fitted findings.

toy_comparison = pd.DataFrame({
    "reference": ["A", "B"],
    "Euclidean_distance": np.sqrt(
        np.sum((toy_references - toy_query[0])**2, axis=1)
    ),
    "weighted_distance": np.sqrt(
        np.sum(toy_weights * (toy_references - toy_query[0])**2, axis=1)
    ),
    "reference_label": toy_labels
})
display(toy_comparison.round(5))
toy_euclidean_prediction, _ = nearest_predict(
    toy_references, toy_labels, toy_query, k=1
)
toy_weighted_prediction, _ = nearest_predict(
    toy_references, toy_labels, toy_query, weights=toy_weights, k=1
)
print("Euclidean copied label:", toy_euclidean_prediction[0])
print("Weighted copied label:", toy_weighted_prediction[0])
print("The unknown query value has not been invented.")

## 2. Turn labelled contracts into a small regression problem

For reference pair \((i,j)\), form the coordinate-wise squared differences

\[
H_{ij,k}=(z_{ik}-z_{jk})^2.
\]

Normalize response gaps using a scale fitted only on reference MC labels:

\[
s=\max\!\left(\max_i\widehat y_i-\min_i\widehat y_i,\ 10^{-6}\right),
\qquad
t_{ij}=\left(\frac{\widehat y_i-\widehat y_j}{s}\right)^2.
\]

The fitted weights solve

\[
\min_{\substack{w\geq0\\\mathbf1^\top w=p}}
\frac1{|\mathcal P|}\sum_{(i,j)\in\mathcal P}
\left(\frac{H_{ij}^\top w}{p}-t_{ij}\right)^2
+\frac{\lambda}{2}\sum_{k=1}^{p}(w_k-1)^2.
\]

The division by \(p\) places the squared distance on a mean-coordinate scale; it does not change nearest-neighbor rankings. There is no intercept: identical feature vectors have distance zero. The response scale is a modelling choice, not an extra free parameter fitted on validation data.

This is a convex quadratic loss on a simplex-like feasible set. The included SLSQP optimizer checks success and feasibility; it returns Euclidean weights if optimization fails. We expose that status instead of silently treating a fallback as successful learning.

The penalty shrinks weights toward the Euclidean baseline. A small \(\lambda\) allows larger changes; a large one discourages them. A lower training pair loss does not guarantee a better retrieved value, so we choose \(\lambda\) using validation valuation error.

**Data split rule.** Split contracts before creating pairs. Reference pairs fit the weights. Validation contracts never enter those pairs. Randomly splitting pairs would let the same contract appear in both training and validation, which would not assess generalization to unseen contracts.

In [ ]:
PORTFOLIO_SEED = 2027
SPLIT_SEED = 2027
COMPRESSOR_SEED = 2027
REFERENCE_MC_SEED = 4101
VALIDATION_MC_SEED = 4102
PAIR_SEED = 6101
N_PORTFOLIO = 1000
N_REFERENCES = 40
N_VALIDATION = 100
REFERENCE_PATHS = 2048
VALIDATION_PATHS = 2048
LAMBDA_GRID = [0.0, 0.001, 0.01, 0.1]

portfolio = make_portfolio(n=N_PORTFOLIO, seed=PORTFOLIO_SEED)
parts = split_portfolio(portfolio, seed=SPLIT_SEED)
candidate = parts["candidate"].copy()
validation = parts["validation"].sample(
    n=min(N_VALIDATION, len(parts["validation"])), random_state=2028
).copy()
# The held-out test split is not accessed or labelled.

t0 = perf_counter()
scaler, Zcandidate, Zvalidation = build_features(candidate, validation)
feature_seconds = perf_counter() - t0

t0 = perf_counter()
reference_positions = select_representatives(
    Zcandidate, m=N_REFERENCES, method="kmeans", seed=COMPRESSOR_SEED
)
compression_seconds = perf_counter() - t0
references = candidate.iloc[reference_positions].copy()
Zreference = Zcandidate[reference_positions]

t0 = perf_counter()
reference_mc = gmab_mc(
    references, n_paths=REFERENCE_PATHS, seed=REFERENCE_MC_SEED
)
reference_mc_seconds = perf_counter() - t0
y_reference = reference_mc["value"].to_numpy()

t0 = perf_counter()
validation_mc = gmab_mc(
    validation, n_paths=VALIDATION_PATHS, seed=VALIDATION_MC_SEED
)
validation_mc_seconds = perf_counter() - t0
y_validation = validation_mc["value"].to_numpy()
premium_validation = validation["premium"].to_numpy()

pair_i, pair_j = np.triu_indices(len(references), k=1)
pair_H = (Zreference[pair_i] - Zreference[pair_j])**2
reference_scale = max(float(np.ptp(y_reference)), 1e-6)
pair_target = ((y_reference[pair_i] - y_reference[pair_j])
               / reference_scale)**2

pair_preview = pd.DataFrame(pair_H[:6], columns=[
    "squared_difference_" + name for name in FEATURES
])
pair_preview["reference_i_id"] = references["contract_id"].to_numpy()[pair_i[:6]]
pair_preview["reference_j_id"] = references["contract_id"].to_numpy()[pair_j[:6]]
pair_preview["squared_normalized_MC_gap"] = pair_target[:6]
display(pair_preview)
print("Reference contracts:", len(references))
print("Distinct unordered pairs available:", len(pair_i))
print("Reference-fitted response scale:", reference_scale)
print("No validation or test response entered pair construction.")

## 3. Why Monte Carlo noise matters for pair labels

Write \(\widehat y_i=y_i+\varepsilon_i\), with
\(E[\varepsilon_i]=0\) and
\(\mathrm{se}_i^2=\operatorname{Var}(\widehat y_i)\).
For independent MC labels from different contracts,

\[
E\!\left[(\widehat y_i-\widehat y_j)^2\right]
=(y_i-y_j)^2+\mathrm{se}_i^2+\mathrm{se}_j^2.
\]

This identity is for **unnormalized** gaps and true MC variances. The reported sample standard errors estimate those variances. For example, a true gap of 0.03 with MC standard errors 0.01 and 0.02 has expected squared observed gap
\(0.03^2+0.01^2+0.02^2=0.0014\), larger than the true squared gap \(0.0009\).

Even two truly equally valued contracts usually have a positive observed squared gap. A flexible weighting rule can partly fit simulation noise.

If a deterministic scale \(s\) is fixed in advance, the identity may be divided by \(s^2\). Here \(s\) is estimated from the same noisy reference labels, so it is random and dependent on the gaps. We cannot obtain an exact expectation formula simply by dividing the identity by the observed \(s^2\). Our fitted pair objective is **not an unbiased loss for true response gaps**.

One possible investigation is subtracting estimated MC variances from an observed squared gap. The result can be negative. Clipping it to zero changes its expectation and introduces bias; this is not an automatic correction implemented by our learner.

Finally, 40 references create 780 pairs, but not 780 independent contracts. Pairs share labels. More pair rows do not create more independently simulated information, and standard errors cannot be calculated as though the pairs were independent observations.

In [ ]:
noise_example = pd.DataFrame([{
    "true_gap_squared": 0.03**2,
    "MC_variance_i": 0.01**2,
    "MC_variance_j": 0.02**2,
    "expected_observed_gap_squared": 0.03**2 + 0.01**2 + 0.02**2
}])
display(noise_example)

raw_pair_gap2 = (y_reference[pair_i] - y_reference[pair_j])**2
reference_se = reference_mc["se"].to_numpy()
estimated_pair_noise = reference_se[pair_i]**2 + reference_se[pair_j]**2
subtracted_gap2 = raw_pair_gap2 - estimated_pair_noise
display(pd.DataFrame([{
    "mean_observed_squared_gap": raw_pair_gap2.mean(),
    "mean_estimated_pair_MC_variance": estimated_pair_noise.mean(),
    "fraction_negative_after_variance_subtraction": np.mean(subtracted_gap2 < 0),
    "reference_count": len(references),
    "pair_count": len(pair_i)
}]).round(7))
print("The metric below still uses the declared raw-gap objective.")

## 4. Select regularization by validation valuation error

For each \(\lambda\), fit \(w_\lambda\) on the same reference pairs. For validation contract \(q\), predict

\[
j^*_\lambda(q)=\arg\min_{j\in\mathcal R}d_{w_\lambda}(q,j),
\qquad
\widehat y_\lambda(q)=\widehat y_{j^*_\lambda(q)}.
\]

Choose the declared grid setting with the smallest RMSE against independent validation MC labels. We use the same pair seed and cap for each fit, so changing \(\lambda\) does not also change the sampled pair design. In this demo all 780 unordered pairs fit below the cap of 2,000; larger designs may sample pairs with replacement.

Keep the validation set out of pair fitting. It can select a trained weight vector, but it must not be used to construct its regression rows or response scale. The test set remains sealed.

The next table includes optimizer status, pair count, and fitting time. If every fit fails, the workflow explicitly uses the Euclidean fallback. A failed optimizer must never disappear from the result table.

Inspect the weight path as well as the smallest validation score. A selected weight is conditional on the reference bank, MC labels, scale, and penalty grid. It is not a universal importance ranking or a claim about the sign of a feature's pricing effect.

In [ ]:
metric_candidates = []
metric_search_start = perf_counter()

for lam in LAMBDA_GRID:
    t0 = perf_counter()
    fitted = fit_metric(
        Zreference, y_reference, lam=lam, max_pairs=2000, seed=PAIR_SEED
    )
    fit_seconds = perf_counter() - t0

    t0 = perf_counter()
    prediction, neighbors = nearest_predict(
        Zreference, y_reference, Zvalidation,
        weights=fitted["weights"], k=1
    )
    prediction_seconds = perf_counter() - t0
    metrics = score_predictions(
        y_validation, prediction, premium=premium_validation
    )
    weights = np.asarray(fitted["weights"])
    row = {
        "lambda": lam, "solver_success": bool(fitted["success"]),
        "pair_count": int(fitted["pair_count"]),
        "objective": float(fitted["objective"]),
        "scale": float(fitted["scale"]),
        "sum_weights": float(weights.sum()),
        "min_weight": float(weights.min()),
        "near_zero_weights": int(np.sum(weights < 1e-7)),
        "validation_RMSE_pp": 100 * metrics["rmse"],
        "fit_seconds": fit_seconds,
        "prediction_seconds": prediction_seconds,
        "solver_message": fitted.get("message", ""),
    }
    metric_candidates.append({
        "row": row, "fit": fitted, "prediction": prediction,
        "neighbors": neighbors, "metrics": metrics
    })

successful_candidates = [
    item for item in metric_candidates if item["fit"]["success"]
]
selection_pool = successful_candidates if successful_candidates else metric_candidates
chosen_metric = min(
    selection_pool,
    key=lambda item: (item["metrics"]["rmse"], item["row"]["lambda"])
)
metric_search_seconds = perf_counter() - metric_search_start
selected_lambda = chosen_metric["row"]["lambda"]
selected_weights = chosen_metric["fit"]["weights"]

metric_table = pd.DataFrame([entry["row"] for entry in metric_candidates])
metric_table["selected"] = metric_table["lambda"] == selected_lambda
display(metric_table.round(6))
if chosen_metric["fit"]["success"]:
    print("Selected lambda on validation data:", selected_lambda)
else:
    print("All metric fits failed; predictions use the explicit Euclidean fallback.")
print("Total metric-grid search seconds:", round(metric_search_seconds, 6))

In [ ]:
weight_path = pd.DataFrame({
    f"lambda={entry['row']['lambda']:g}": entry["fit"]["weights"]
    for entry in metric_candidates
}, index=FEATURES)
display(weight_path.round(5))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
axes[0].bar(FEATURES, selected_weights, color="#37728e")
axes[0].axhline(1.0, color="#9d5239", linestyle="--", label="Euclidean weight")
axes[0].set(title=f"Selected weights, lambda={selected_lambda:g}",
            ylabel="Nonnegative weight; total equals feature count")
axes[0].tick_params(axis="x", rotation=40)
axes[0].legend()

p = len(FEATURES)
euclidean_pair_distance = pair_H @ np.ones(p) / p
learned_pair_distance = pair_H @ selected_weights / p
axes[1].scatter(pair_target, euclidean_pair_distance, s=11,
                alpha=0.25, color="#758b9a", label="Euclidean")
axes[1].scatter(pair_target, learned_pair_distance, s=11,
                alpha=0.25, color="#b96642", label="Learned")
axes[1].plot([0, 1], [0, 1], "--", color="#666666", linewidth=1)
axes[1].set(xlabel="Squared normalized reference MC gap",
            ylabel="Weighted squared feature distance / p",
            title="Training-pair diagnostic; not validation evidence")
axes[1].legend()
fig.tight_layout()
plt.show()

## 5. Compare retrieval with direct prediction

The comparison now has three kinds of method:

| Method | Fitted object | Prediction |
|---|---|---|
| Euclidean 1NN | No response-based geometry | Copy one nearest reference label |
| Uniform 3NN | No response-based geometry | Average three nearest reference labels |
| Learned 1NN | Diagonal feature weights | Copy one nearest reference under learned distance |
| Ridge | Regression coefficients | Evaluate a fitted linear function |
| Random forest | An ensemble of trees | Average tree predictions |

Every row uses the same reference contracts and MC labels. The learned metric receives a four-setting penalty search; Ridge and random forest receive the same three-setting grids used in Week 5. Nearest baselines keep declared fixed settings. Record these differences in search effort and charge their costs.

For 1NN the observed validation residual decomposes exactly as

\[
\widehat y_{j^*(q)}-\widehat y^{\,\mathrm{MC}}_{\mathrm{val},q}
=
\underbrace{y_{j^*(q)}-y_q}_{\text{true transfer mismatch}}
+\underbrace{\varepsilon_{j^*(q)}}_{\text{reference MC error}}
-\underbrace{\varepsilon_{\mathrm{val},q}}_{\text{validation MC error}}.
\]

The terms explain why a geometrically plausible neighbor can still predict poorly. When weights are fitted from noisy labels, the selected neighbor also depends on those labels; we cannot assume the selected reference error remains unbiased conditional on that selection.

A response gap measured against the validation MC mean is useful for inspecting a neighbor, but for 1NN it is exactly the absolute validation prediction error. It is not a second independent accuracy measure.

In [ ]:
comparison_predictions = {}
comparison_rows = []
neighbor_indices = {}
shared_development_seconds = (
    feature_seconds + compression_seconds
    + reference_mc_seconds + validation_mc_seconds
)

def comparison_row(name, pred, settings, candidates, search_seconds,
                   chosen_fit_seconds, chosen_prediction_seconds):
    metric = score_predictions(y_validation, pred, premium=premium_validation)
    return {
        "method": name, "settings": settings,
        "validation_RMSE_pp": 100 * metric["rmse"],
        "validation_MAE_pp": 100 * metric["mae"],
        "validation_p95_abs_pp": 100 * metric["p95_abs"],
        "validation_bias_pp": 100 * metric["bias"],
        "validation_aggregate_error_pct": 100 * metric["portfolio_relative_error"],
        "tuning_candidates": candidates,
        "chosen_fit_seconds": chosen_fit_seconds,
        "chosen_prediction_seconds": chosen_prediction_seconds,
        "search_seconds": search_seconds,
        "standalone_development_seconds": shared_development_seconds + search_seconds,
    }

for name, k in [("Euclidean1NN", 1), ("kNN3", 3)]:
    t0 = perf_counter()
    pred, indices = nearest_predict(
        Zreference, y_reference, Zvalidation, k=k
    )
    elapsed = perf_counter() - t0
    comparison_predictions[name] = pred
    neighbor_indices[name] = indices
    comparison_rows.append(comparison_row(
        name, pred, f"k={k}", 1, elapsed, 0.0, elapsed
    ))

comparison_predictions["Learned1NN"] = chosen_metric["prediction"]
neighbor_indices["Learned1NN"] = chosen_metric["neighbors"]
metric_status = ("fitted" if chosen_metric["fit"]["success"]
                 else "Euclidean fallback")
comparison_rows.append(comparison_row(
    "Learned1NN", chosen_metric["prediction"],
    f"lambda={selected_lambda:g}; {metric_status}",
    len(LAMBDA_GRID), metric_search_seconds,
    chosen_metric["row"]["fit_seconds"],
    chosen_metric["row"]["prediction_seconds"]
))

direct_models = make_regressors(seed=2027)
direct_grids = {
    "Ridge": [{"alpha": a} for a in [0.1, 1.0, 10.0]],
    "RandomForest": [{"min_samples_leaf": leaf} for leaf in [1, 3, 5]]
}
direct_trial_rows = []
direct_selected = {}

for name, grid in direct_grids.items():
    search_start = perf_counter()
    candidates = []
    for parameters in grid:
        estimator = clone(direct_models[name]).set_params(**parameters)
        t0 = perf_counter()
        estimator.fit(Zreference, y_reference)
        fit_seconds = perf_counter() - t0
        t0 = perf_counter()
        pred = estimator.predict(Zvalidation)
        prediction_seconds = perf_counter() - t0
        metric = score_predictions(y_validation, pred)
        trial = {
            "method": name, "parameters": json.dumps(parameters, sort_keys=True),
            "validation_RMSE_pp": 100 * metric["rmse"],
            "fit_seconds": fit_seconds, "prediction_seconds": prediction_seconds,
        }
        direct_trial_rows.append(trial)
        candidates.append({
            "row": trial, "estimator": estimator, "prediction": pred,
            "rmse": metric["rmse"]
        })
    chosen = min(candidates, key=lambda item: (item["rmse"], item["row"]["parameters"]))
    search_seconds = perf_counter() - search_start
    direct_selected[name] = chosen
    comparison_predictions[name] = chosen["prediction"]
    comparison_rows.append(comparison_row(
        name, chosen["prediction"], chosen["row"]["parameters"], len(grid),
        search_seconds, chosen["row"]["fit_seconds"],
        chosen["row"]["prediction_seconds"]
    ))

validation_comparison = pd.DataFrame(comparison_rows).sort_values(
    "validation_RMSE_pp", kind="stable"
).reset_index(drop=True)
display(pd.DataFrame(direct_trial_rows).round(6))
display(validation_comparison[[
    "method", "settings", "validation_RMSE_pp", "validation_MAE_pp",
    "validation_p95_abs_pp", "validation_bias_pp",
    "validation_aggregate_error_pct"
]].round(5))
print("These scores guide development. They are not final test estimates.")

In [ ]:
euclidean_neighbor = neighbor_indices["Euclidean1NN"][:, 0]
learned_neighbor = neighbor_indices["Learned1NN"][:, 0]
reference_ids = references["contract_id"].to_numpy()
neighbor_audit = pd.DataFrame({
    "query_contract_id": validation["contract_id"].to_numpy(),
    "query_premium": premium_validation,
    "validation_MC": y_validation,
    "validation_MC_SE": validation_mc["se"].to_numpy(),
    "Euclidean_reference_id": reference_ids[euclidean_neighbor],
    "learned_reference_id": reference_ids[learned_neighbor],
    "Euclidean_reference_MC": y_reference[euclidean_neighbor],
    "learned_reference_MC": y_reference[learned_neighbor],
    "Euclidean_response_gap_pp": 100 * np.abs(
        y_reference[euclidean_neighbor] - y_validation
    ),
    "learned_response_gap_pp": 100 * np.abs(
        y_reference[learned_neighbor] - y_validation
    ),
})
neighbor_audit["neighbor_changed"] = (
    neighbor_audit["Euclidean_reference_id"]
    != neighbor_audit["learned_reference_id"]
)
neighbor_audit["gap_change_pp"] = (
    neighbor_audit["learned_response_gap_pp"]
    - neighbor_audit["Euclidean_response_gap_pp"]
)
neighbor_audit["learned_currency_prediction"] = (
    neighbor_audit["query_premium"] * neighbor_audit["learned_reference_MC"]
)

print("Fraction of validation queries selecting a different reference:",
      neighbor_audit["neighbor_changed"].mean())
ordered_cases = neighbor_audit.sort_values("gap_change_pp")
case_study = pd.concat([ordered_cases.head(3), ordered_cases.tail(3)]).drop_duplicates(
    "query_contract_id"
)
display(case_study.round(6))
print("Negative gap_change_pp means smaller observed error for learned 1NN.")
print("This case table shows both ends of the observed change, not only favorable cases.")

## 6. What the learned weights do—and what they cannot establish

A large weight makes a difference in that scaled coordinate more influential in retrieval. It does not establish that changing the underlying feature causes a large value change, and it does not encode the sign of the response. The outcome also depends on feature ranges, correlations, reference coverage, the random label scale, and MC noise.

A diagonal distance is intentionally limited. It cannot rotate the feature space, model location-dependent sensitivity, or represent every interaction. Two contracts can have nearly equal liabilities for different reasons, while being far apart in every original coordinate.

The scale constraint prevents arbitrary rescaling, and the penalty discourages unstable deviations from Euclidean weights. Neither creates a theorem that learned similarity will outperform ordinary distance. A meaningful project result may show improvement only in certain regimes, no clear difference, or a failure case with a useful explanation.

**Complete these interpretation prompts**

1. Did weights materially change the retrieved reference IDs? Report the fraction.
2. Did any changed neighbors improve validation errors while others worsened them?
3. Does a lower fitted pair loss always coincide with lower validation RMSE across the penalty grid?
4. Which conclusion survives comparison with direct value prediction?
5. Which conclusions need repeated seeds and the later frozen test assessment?

The pair construction and optimization are real computation. Their cost belongs in the learned method's ledger even though they do not require new simulation labels.

In [ ]:
display(validation_comparison[[
    "method", "tuning_candidates", "chosen_fit_seconds",
    "chosen_prediction_seconds", "search_seconds",
    "standalone_development_seconds"
]].round(6))

simulation_ledger = pd.DataFrame([
    {"bank": "References, shared",
     "contract_valuations": len(references),
     "payoff_paths": len(references) * REFERENCE_PATHS,
     "simulation_seconds": reference_mc_seconds},
    {"bank": "Validation, shared",
     "contract_valuations": len(validation),
     "payoff_paths": len(validation) * VALIDATION_PATHS,
     "simulation_seconds": validation_mc_seconds}
])
display(simulation_ledger.round(6))
print("Actual guided simulation payoff paths:",
      int(simulation_ledger["payoff_paths"].sum()))
print("Metric fitting calls:", len(LAMBDA_GRID))
print("Pair rows processed across metric fitting calls:",
      sum(item["fit"]["pair_count"] for item in metric_candidates))
print("These pair rows reuse", len(references), "independent reference labels.")
print("Feature preparation seconds:", round(feature_seconds, 6))
print("Reference selection seconds:", round(compression_seconds, 6))

fig, ax = plt.subplots(figsize=(8.2, 4.4))
names = validation_comparison["method"].tolist()
ax.bar(names, validation_comparison["validation_RMSE_pp"],
       color=["#39758f" if name != "Learned1NN" else "#b55e3c" for name in names])
ax.set(ylabel="Validation RMSE (pp of premium)",
       title="Development comparison on one shared MC label design")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
plt.show()

**Cost interpretation.** The standalone column includes shared feature preparation, compression, reference MC, validation MC, and that method's search. The metric's search includes pair construction inside each fit, constrained optimization, validation prediction, and scoring. Display and plotting time are teaching overhead outside these method timings. The nearest baselines record their direct prediction time; their reporting overhead is negligible at this scale.

All methods reuse the same simulation banks. Count those banks once when reporting the actual multi-model campaign, but charge them in every standalone workflow comparison. As in Weeks 4–5, this is a same-label educational comparison; the final protocol will also compare methods within a declared total development simulation budget.

**Research note:** write 250 words that give your selected penalty and solver status, one observation about the weights, one quantitative pricing comparison, the simulation/fitting cost, and one limitation. A conclusion that the metric did not clearly help is acceptable if supported by the evidence.

## Reading assignment — approximately 45 minutes

1. **Ben Mingbin Feng, Zhenni Tan, and Jiayi Zheng (2020), _Efficient Simulation Designs for Valuation of Large Variable Annuity Portfolios_.** [Author PDF](https://www.math.uwaterloo.ca/~mbfeng/papers/2020_VAPort.pdf); [DOI](https://doi.org/10.1080/10920277.2019.1685394). Revisit Section 5, Equation (12), and identify which part of the prediction rule changes when we learn the distance.
2. **Hao Quan and Ben (Mingbin) Feng (2026), _Warm-Started Projected Robbins–Monro for Portfolio-Scale Variable Annuity Fee Setting_.** Working manuscript supplied by the mentor. Read the introduction, the valuation setup, and the intuitive explanation of the start-value approximation. The proofs and stochastic approximation algorithm are not required.

**Focused questions**

- How does replacing the distance leave the nearest-reference transfer formula otherwise unchanged?
- What is our prediction target, and how does it differ from a fair-fee root or a warm start for root finding?
- Why can squared differences of MC labels exaggerate true response differences?
- Why should the contract split happen before pair generation?
- What information would you need before claiming that a fitted feature weight generalizes to another portfolio?

The diagonal learner is a self-contained teaching construction developed in this notebook; no contract-similarity research manuscript is required.

## Homework 1 — A response-label negative control

**Core; about 45–60 minutes.** Randomly permute reference responses **only when fitting the metric**. Continue transferring the original, correctly aligned MC reference values at prediction time. This isolates whether the response information used to learn the geometry is useful.

Keep the selected penalty fixed and record the shuffle seed. Compare Euclidean 1NN, the original learned metric, and the shuffled-label metric on the same validation set. Repeat for three shuffle seeds. Explain why one unfavorable or favorable shuffle is not a formal statistical conclusion.

Do not tune the negative control repeatedly until it produces the result you hoped to see.

In [ ]:
shuffle_seed = 6201
shuffle_rng = np.random.default_rng(shuffle_seed)
shuffled_fit_labels = shuffle_rng.permutation(y_reference)
shuffled_metric = fit_metric(
    Zreference, shuffled_fit_labels, lam=selected_lambda,
    max_pairs=2000, seed=PAIR_SEED
)
# Prediction still copies the ORIGINAL correctly aligned reference labels.
shuffled_prediction, _ = nearest_predict(
    Zreference, y_reference, Zvalidation,
    weights=shuffled_metric["weights"], k=1
)
negative_control_rows = []
for name, pred in [
    ("Euclidean1NN", comparison_predictions["Euclidean1NN"]),
    ("Original learned metric", comparison_predictions["Learned1NN"]),
    ("Shuffled-fit-label metric", shuffled_prediction)
]:
    negative_control_rows.append({
        "method": name, "fixed_lambda": selected_lambda,
        "validation_RMSE_pp": 100 * score_predictions(
            y_validation, pred
        )["rmse"]
    })
display(pd.DataFrame(negative_control_rows).round(6))
print("Shuffled metric solver success:", shuffled_metric["success"])
display(pd.DataFrame({
    "feature": FEATURES, "original_weights": selected_weights,
    "shuffled_fit_weights": shuffled_metric["weights"]
}))

## Homework 2 — Change label precision without changing contracts

**Core; about 60 minutes.** Fit the same metric using 1,024 and 2,048 paths per reference, while keeping the reference contracts, validation labels, pair design, and penalty fixed. Compare reference standard errors, fitted weights, and validation RMSE.

The helper's fixed seed makes the smaller path stream a prefix of the larger one, so the two label estimates are correlated. This is a paired exploratory design, not two independent repetitions. For an additional repetition, change the reference experiment seed and report that change.

The 2,048-path bank is reused below. Report both each design's reference budget and the actual extra simulation performed by your homework. Do not use the exact-price function to choose the preferred design.

In [ ]:
precision_rows = []
precision_weights = {}
for paths in [1024, 2048]:
    reuse_existing = paths == REFERENCE_PATHS
    bank = (reference_mc if reuse_existing else gmab_mc(
        references, n_paths=paths, seed=REFERENCE_MC_SEED
    ))
    precision_labels = bank["value"].to_numpy()
    fitted = fit_metric(
        Zreference, precision_labels, lam=selected_lambda,
        max_pairs=2000, seed=PAIR_SEED
    )
    pred, _ = nearest_predict(
        Zreference, precision_labels, Zvalidation,
        weights=fitted["weights"], k=1
    )
    precision_weights[f"{paths}_paths"] = fitted["weights"]
    precision_rows.append({
        "paths_per_reference": paths,
        "reference_budget_paths": len(references) * paths,
        "extra_simulated_paths_in_this_cell": (
            0 if reuse_existing else len(references) * paths
        ),
        "mean_reference_SE_pp": 100 * bank["se"].mean(),
        "solver_success": fitted["success"],
        "validation_RMSE_pp": 100 * score_predictions(y_validation, pred)["rmse"]
    })
display(pd.DataFrame(precision_rows).round(6))
display(pd.DataFrame(precision_weights, index=FEATURES).round(6))

## Homework 3 — Combine the learned distance with averaging (optional)

Use the selected weights with \(k=1,3,5\) and average the selected labels. Compare to Euclidean \(k=1,3,5\) on the same bank. This changes the prediction rule while holding the geometry fixed within each group.

A larger \(k\) can average reference MC noise but may include contracts with different true values. Report both possibilities in your interpretation. Count this additional validation search and reserve any final claim for the frozen assessment.

In [ ]:
hybrid_rows = []
for geometry, weights in [
    ("Euclidean", np.ones(len(FEATURES))),
    ("Learned", selected_weights)
]:
    for k in [1, 3, 5]:
        pred, _ = nearest_predict(
            Zreference, y_reference, Zvalidation, weights=weights, k=k
        )
        hybrid_rows.append({
            "geometry": geometry, "neighbors": k,
            "validation_RMSE_pp": 100 * score_predictions(
                y_validation, pred
            )["rmse"]
        })
display(pd.DataFrame(hybrid_rows).round(6))

## Weekly deliverable checklist

- [ ] I can explain the weighted distance and the sum-of-weights constraint.
- [ ] I use “pseudometric” when zero weights or feature-identical contracts permit zero distance.
- [ ] My pair features, pair responses, and response scale use reference contracts only.
- [ ] I can state the unnormalized MC squared-gap identity and its assumptions.
- [ ] I do not treat shared-contract pairs as independent observations.
- [ ] My penalty table includes every attempted fit and any optimizer fallback.
- [ ] I choose the penalty with validation MC labels and keep the test set closed.
- [ ] I compare learned retrieval with Euclidean neighbors and direct regression.
- [ ] My case study shows favorable and unfavorable changes when both occur.
- [ ] I report label generation, pair fitting, tuning, and prediction work.
- [ ] I completed the reading questions and the two core exercises.
- [ ] My claims concern this fixed-fee top-up task and do not imply a new fair-fee algorithm.

**Bring to the meeting:** your metric table, weight plot, negative-control results, a 250-word research note, and one small extension you would like to preregister during Week 7's benchmark design.